<hr style="height:1rem;">

## Imports

In [1]:
import uuid
import time
import numpy as np
import pandas as pd

import re
import json
import os
from pathlib import Path

import openai
from openai import OpenAI # for calling the OpenAI API

<hr style="height:1rem;">

## Path Settings

In [2]:
CURRENT_PATH = os.getcwd()
CURRENT_DATA_PATH = os.path.join(
    CURRENT_PATH,
    'data'
)
CURRENT_RESULTS_PATH = os.path.join(
    CURRENT_PATH,
    'results'
)
NOTEBOOK_PATH = Path.cwd().resolve().parents[0]
NOTEBOOK_EVALUATION_PATH = os.path.join(
    NOTEBOOK_PATH,
    'evaluation'
)
NOTEBOOK_RESULTS_PATH = os.path.join(
    NOTEBOOK_PATH,
    'results'
)
DISSERTATION_PATH = Path.cwd().resolve().parents[1]
DISSERTATION_DATA_PATH = os.path.join(
    DISSERTATION_PATH,
    'data'
)
PROJECT_PATH = Path.cwd().resolve().parents[2]
PROJECT_DATA_PATH = os.path.join(
    PROJECT_PATH,
    'data'
)

print(CURRENT_PATH)
print(CURRENT_DATA_PATH)
print(CURRENT_RESULTS_PATH)
print(NOTEBOOK_PATH)
print(NOTEBOOK_RESULTS_PATH)
print(NOTEBOOK_EVALUATION_PATH)
print(DISSERTATION_PATH)
print(DISSERTATION_DATA_PATH)
print(PROJECT_PATH)
print(PROJECT_DATA_PATH)

D:\ODU\03 - Dissertation Defense\003_dissertation\notebook\06_knowledge_graph
D:\ODU\03 - Dissertation Defense\003_dissertation\notebook\06_knowledge_graph\data
D:\ODU\03 - Dissertation Defense\003_dissertation\notebook\06_knowledge_graph\results
D:\ODU\03 - Dissertation Defense\003_dissertation\notebook
D:\ODU\03 - Dissertation Defense\003_dissertation\notebook\results
D:\ODU\03 - Dissertation Defense\003_dissertation\notebook\evaluation
D:\ODU\03 - Dissertation Defense\003_dissertation
D:\ODU\03 - Dissertation Defense\003_dissertation\data
D:\ODU\03 - Dissertation Defense
D:\ODU\03 - Dissertation Defense\data


<hr style="height:1rem;">

## Variable Definitions

In [3]:
GPT_SECRET_KEY_FILE =  os.path.join(
    PROJECT_DATA_PATH,
    'GPT_SECRET_KEY.json'
)

In [4]:
with open(GPT_SECRET_KEY_FILE) as f:
    data = json.load(f)
    API_KEY = data["API_KEY"]

In [5]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=API_KEY,
)

<hr style="height:1rem;">

## OPENAI's PARAMETER Definitions

##### MODEL
model used for decoding

[Model Overview](https://platform.openai.com/docs/models/overview)

[Chat Completions API](https://platform.openai.com/docs/guides/text-generation/chat-completions-api)

ID of the model to use. You can use the [List models](https://platform.openai.com/docs/api-reference/models/list) API to see all of your available models, or see our Model overview for descriptions of them.

API_ENDPOINT: 0 - Completion

API_ENDPOINT: 1 - ChatCompletion

<hr style="height:0.25rem;">

##### MAX_LENGTH (Integer): Defaults to 16

Limiting the Response Length.  The maximum length of output tokens by model for reasoning extraction

The maximum number of tokens that can be generated in the completion.

The token count of your prompt plus max_tokens cannot exceed the model's context length. Example Python code for counting tokens.

In [6]:
MAX_LENGTH = 256

<hr style="height:0.25rem;">

##### SEED

Set the seed parameter to any integer of your choice and use the same value across requests you'd like deterministic outputs for.

If specified, our system will make a best effort to sample deterministically, such that repeated requests with the same seed and parameters should return the same result.

Determinism is not guaranteed, and you should refer to the system_fingerprint response parameter to monitor changes in the backend.

In [1]:
SEED = 123

<hr style="height:0.25rem;">

##### TEMPERATURE (0.2 to 0.8)

Lower values for temperature result in more consistent outputs (e.g. 0.2), while higher values generate more diverse and creative results (e.g. 1.0). Select a temperature value based on the desired trade-off between coherence and creativity for your specific application. The temperature can range is from 0 to 2.

What sampling temperature to use, between 0 and 2. Higher values like 0.8 will make the output more random, while lower values like 0.2 will make it more focused and deterministic.

We generally recommend altering this or top_p but not both.

In [8]:
#TEMPERATURE = 0.2
TEMPERATURE = 0

<hr style="height:0.25rem;">

##### TOP_P (0.3 to 0.9)
Top P (Nucleus Sampling): Controlling Response Quality

An alternative to sampling with temperature, called nucleus sampling, where the model considers the results of the tokens with top_p probability mass. So 0.1 means only the tokens comprising the top 10% probability mass are considered.

We generally recommend altering this or temperature but not both.

In [9]:
#TOP_P = 1
TOP_P = 0.000001

<hr style="height:0.25rem;">

##### FREQUENCY_PENALTY (0.6 to 1): Defaults to 0
Frequency Penalty: Controlling Repetitive Responses 

Number between -2.0 and 2.0. Positive values penalize new tokens based on their existing frequency in the text so far, decreasing the model's likelihood to repeat the same line verbatim.

In [10]:
FREQUENCY_PENALTY = 0

<hr style="height:0.25rem;">

##### PRESENCE_PENALTY (0.2 to 0.6): Defaults to 0
Presence Penalty: Controlling Avoidance of Certain Topics 

Number between -2.0 and 2.0. Positive values penalize new tokens based on whether they appear in the text so far, increasing the model's likelihood to talk about new topics.

In [11]:
PRESENCE_PENALTY = 0

<hr style="height:0.25rem;">

##### OPEN AI MODEL

In [12]:
GPT4_MODEL = "gpt-4"
GPT4O_MODEL = "gpt-4o"
MODEL = GPT4_MODEL
print(MODEL)

gpt-4


<hr style="height:1rem;">

## Function Definitions

In [13]:
def cot(messages, MODEL):    
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=MAX_LENGTH,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        frequency_penalty=FREQUENCY_PENALTY,
        presence_penalty=PRESENCE_PENALTY,
        seed=SEED        
    )
    # Extracting the text from the response
    result = response.choices[0].message.content
    
    return result, response

<hr style="height:1rem;">

## Read Data into DataFrame

In [14]:
FILENAME = 'ICD10_CODES_NOTES_1000.csv'
FILE = os.path.join(
    NOTEBOOK_RESULTS_PATH,
    FILENAME
)
df = pd.read_csv(FILE)
len(df)
df.head(5)

,article_id,text_references,icd_codes,text,prompt_template
0,S2340-98942015000100005-1,metastatic liver lesion; vomiting; disease; re...,C78.7 R11.10 R69 0DTP 0FB0 R06.00 C56.2 R97.1 R55,We report the case of a 62-year-old patient wh...,\nYou are an expert in medical NLP and knowled...
1,S2254-28842014000200009-1,anemia; deliveries; Caesarean section; tubal l...,D64.9 10E0XZZ 10D00Z1 0UL7 R53.1 R63.0 R63.4 K...,A 33-year-old woman presented to the emergency...,\nYou are an expert in medical NLP and knowled...
2,S2254-28842012000300010-1,renal insufficiency; ischemic colitis; infecti...,N19 K55.9 B99.9 B96.20 K63.5 I82.210 R57.0 R58...,A 72-year-old male in HD program from December...,\nYou are an expert in medical NLP and knowled...
3,S1889-836X2016000100006-1,osteoporosis; otosclerosis; elbow fractures; s...,M81.0 H80.90 S42.409 S42.90X Z87.81 R69 S43.40...,A 65-year-old male patient diagnosed with OI f...,\nYou are an expert in medical NLP and knowled...
4,S1889-836X2015000200005-1,MRI feet; x-ray of feet; MRI of feet; feet ult...,BQ3M BQ0MZZZ BQ3L BH48ZZZ M84.30X R52 R60.9 BQ...,We describe the case of a 47-year-old female p...,\nYou are an expert in medical NLP and knowled...


<hr style="height:1rem;">

## Experiment

In [15]:
all_response_contents = []
all_system_fingerprints = []
all_prompt_tokens = []
all_completion_tokens = []

TOTAL_RUNS = 100
PRINT_STEP = 10
COUNT = 0

TEST_1_CASE = True

In [16]:
for index, row in df.iterrows():
    prompt = str(row["prompt_template"])

    messages = [{
        "role": "user", "content": prompt
    }]
    
    result, response = cot(
        messages,
        GPT4_MODEL
    )

    response_content = response.choices[0].message.content
    system_fingerprint = response.system_fingerprint
    prompt_tokens = response.usage.prompt_tokens
    completion_tokens = response.usage.total_tokens - response.usage.prompt_tokens   

    all_response_contents.append(response_content)
    all_system_fingerprints.append(system_fingerprint)
    all_prompt_tokens.append(prompt_tokens)
    all_completion_tokens.append(completion_tokens)

    if TEST_1_CASE:
        # if set to True, break out of loop.
        break
    
    COUNT = index+1
    if COUNT % PRINT_STEP == 0:   
        print(f"Completed {COUNT} prompts")

<hr style="height:0.25rem;">

### Create Data Frame

In [17]:
save_results_df = pd.DataFrame({
    "response_content": all_response_contents,
    "system_fingerprint": all_system_fingerprints,
    "prompt_tokens": all_prompt_tokens,
    "completion_tokens": all_completion_tokens
})

In [18]:
save_results_df.head(5)

,response_content,system_fingerprint,prompt_tokens,completion_tokens
0,"### **Output:**\n\n```json\n{\n ""nodes"": [\n ...",None,1342,256


<hr style="height:0.25rem;">

### Save the Results

In [20]:
TIMESTR = time.strftime("%Y%m%d-%H%M%S")
OUTPUT_FILENAME = 'KNOWLEDGE_GRAPH_RESULTS_'+TIMESTR+'.csv'
OUTPUT_FILE = os.path.join(
    CURRENT_RESULTS_PATH,
    OUTPUT_FILENAME
)
save_results_df.to_csv(
    OUTPUT_FILE,
    index=False
)